# Course 17: Build Generative AI Applications on Google Cloud
## GCP Machine Learning Engineer Certification

This notebook covers hands-on exercises for building GenAI applications on GCP:
- Gemini prompt design patterns (zero-shot, few-shot, system instructions)
- Function calling: define tools, automatic invocation
- Multimodal: analyze images with Gemini
- RAG demo: embed documents, retrieve, generate answer
- Grounding with Google Search
- Vertex AI Agent Builder concepts (SDK setup)
- Deploy a simple GenAI endpoint on Cloud Run (Dockerfile example)

**Prerequisites**: A GCP project with Vertex AI API enabled and billing configured.

---
## 1. Environment Setup

In [ ]:
# Install required packages
!pip install -q google-cloud-aiplatform>=1.60.0 google-generativeai>=0.5.0 chromadb>=0.4.0

In [ ]:
# Authenticate (Colab)
from google.colab import auth
auth.authenticate_user()

# Set your project and location
PROJECT_ID = "your-project-id"  # <-- Replace with your GCP project ID
LOCATION = "us-central1"

import vertexai
vertexai.init(project=PROJECT_ID, location=LOCATION)

print(f"Vertex AI initialized: project={PROJECT_ID}, location={LOCATION}")

In [ ]:
# Import core classes
from vertexai.generative_models import (
    GenerativeModel, GenerationConfig, Part, Tool,
    FunctionDeclaration, SafetySetting, HarmCategory, HarmBlockThreshold
)

print("Imports successful.")

---
## 2. Prompt Design Patterns

In [ ]:
# --- SYSTEM INSTRUCTIONS ---
# System instructions define the model's persona and constraints

model_with_system = GenerativeModel(
    "gemini-1.5-pro",
    system_instruction="""You are a Google Cloud architecture advisor.
    Rules:
    - Always recommend GCP services specifically.
    - Respond in bullet points.
    - Include pricing tier when relevant.
    - Never suggest non-GCP alternatives."""
)

response = model_with_system.generate_content(
    "I need to build a real-time data pipeline for IoT sensor data."
)
print("=== System Instruction Response ===")
print(response.text)

In [ ]:
# --- ZERO-SHOT ---
# No examples provided; model relies on pre-trained knowledge

model = GenerativeModel("gemini-1.5-pro")

zero_shot_prompt = """Classify this support ticket as BILLING, TECHNICAL, or GENERAL.

Ticket: "My Cloud Function keeps timing out after 60 seconds even though I set the timeout to 540s."

Category:"""

response_zero = model.generate_content(zero_shot_prompt)
print("=== Zero-Shot ===")
print(response_zero.text)

In [ ]:
# --- FEW-SHOT ---
# Provide examples to guide output format

few_shot_prompt = """Classify each support ticket as BILLING, TECHNICAL, or GENERAL.

Ticket: "How do I update my payment method?"
Category: BILLING

Ticket: "My VM instance won't start after the maintenance window."
Category: TECHNICAL

Ticket: "Where can I find documentation for Cloud Run?"
Category: GENERAL

Ticket: "I'm getting a 403 permission denied error when accessing my BigQuery dataset."
Category:"""

response_few = model.generate_content(few_shot_prompt)
print("=== Few-Shot ===")
print(response_few.text)

In [ ]:
# --- CHAIN-OF-THOUGHT ---
# Ask the model to reason step by step

cot_prompt = """A company runs 3 Vertex AI endpoints:
- Endpoint A: 2 replicas, n1-standard-4 ($0.19/hr each)
- Endpoint B: 1 replica, n1-standard-8 ($0.38/hr)
- Endpoint C: 4 replicas, n1-standard-2 ($0.095/hr each)

All endpoints run 24/7 for 30 days. Which endpoint costs the most per month?

Think step by step, showing your calculations for each endpoint before giving the final answer."""

response_cot = model.generate_content(cot_prompt)
print("=== Chain-of-Thought ===")
print(response_cot.text)

In [ ]:
# --- OUTPUT FORMATTING: JSON ---
# Force structured JSON output

json_prompt = """Analyze this GCP error and return ONLY a valid JSON object:

Error: "Cloud Function 'processOrder' exceeded memory limit. Function was allocated 256MB.
Peak usage: 512MB. The function was terminated."

JSON keys:
- "service": the GCP service
- "error_type": OOM, TIMEOUT, PERMISSION, or OTHER
- "severity": HIGH, MEDIUM, or LOW
- "recommendation": one-sentence fix

Return ONLY valid JSON:"""

config = GenerationConfig(temperature=0.0, max_output_tokens=200)
response_json = model.generate_content(json_prompt, generation_config=config)
print("=== JSON Output ===")
print(response_json.text)

# Parse to verify it's valid JSON
import json
try:
    parsed = json.loads(response_json.text.strip().strip('```json').strip('```'))
    print("\nParsed successfully:", parsed)
except json.JSONDecodeError as e:
    print(f"\nJSON parse error: {e}")

### Prompt Design Summary

| Technique | When to Use | Token Cost | Accuracy |
|-----------|------------|------------|----------|
| **System Instructions** | Persistent persona/rules | Low (set once) | High for consistency |
| **Zero-shot** | Simple, obvious tasks | Lowest | Moderate |
| **Few-shot** | Format-specific outputs | Medium | High |
| **Chain-of-thought** | Reasoning/math/logic | Higher | Highest for complex tasks |
| **Output formatting** | Structured data extraction | Low overhead | High with temp=0 |

---
## 3. Function Calling

In [ ]:
# Define tools with function declarations

get_weather = FunctionDeclaration(
    name="get_current_weather",
    description="Get the current weather for a given location",
    parameters={
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "City and state, e.g. San Francisco, CA"
            },
            "unit": {
                "type": "string",
                "enum": ["celsius", "fahrenheit"],
                "description": "Temperature unit"
            }
        },
        "required": ["location"]
    }
)

search_products = FunctionDeclaration(
    name="search_products",
    description="Search for products in the inventory database",
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Product search query"
            },
            "max_results": {
                "type": "integer",
                "description": "Maximum number of results to return"
            }
        },
        "required": ["query"]
    }
)

# Create a tool with both function declarations
tool = Tool(function_declarations=[get_weather, search_products])

print("Tools defined:")
print(f"  - get_current_weather: Get weather for a location")
print(f"  - search_products: Search product inventory")

In [ ]:
# Send a query that should trigger function calling

fc_model = GenerativeModel("gemini-1.5-pro", tools=[tool])

response = fc_model.generate_content(
    "What's the weather like in Tokyo right now?"
)

# Check if the model returned a function call
candidate = response.candidates[0]
part = candidate.content.parts[0]

if hasattr(part, 'function_call') and part.function_call.name:
    fc = part.function_call
    print("=== Function Call Detected ===")
    print(f"Function: {fc.name}")
    print(f"Arguments: {dict(fc.args)}")
else:
    print("No function call - model responded with text:")
    print(part.text)

In [ ]:
# Complete the function calling round-trip
# 1. Model suggests a function call
# 2. We execute the function (simulated here)
# 3. We send the result back to the model
# 4. Model generates a natural language response

from vertexai.generative_models import Content, Part
from google.protobuf.struct_pb2 import Struct

# Simulate the function execution result
weather_result = {
    "location": "Tokyo",
    "temperature": 22,
    "unit": "celsius",
    "condition": "Partly cloudy",
    "humidity": 65
}

# Build the function response
function_response_part = Part.from_function_response(
    name="get_current_weather",
    response={"result": weather_result}
)

# Send function result back to model for final response
final_response = fc_model.generate_content(
    [
        Content(role="user", parts=[Part.from_text("What's the weather like in Tokyo right now?")]),
        Content(role="model", parts=[part]),  # The function call from the model
        Content(role="user", parts=[function_response_part]),  # Our function result
    ]
)

print("=== Final Response (after function execution) ===")
print(final_response.text)

In [ ]:
# Test with a product search query

response2 = fc_model.generate_content(
    "Find me the top 3 wireless headphones under $100"
)

candidate2 = response2.candidates[0]
part2 = candidate2.content.parts[0]

if hasattr(part2, 'function_call') and part2.function_call.name:
    fc2 = part2.function_call
    print("=== Function Call Detected ===")
    print(f"Function: {fc2.name}")
    print(f"Arguments: {dict(fc2.args)}")
else:
    print("Text response:", part2.text)

---
## 4. Multimodal: Analyze an Image with Gemini

In [ ]:
import requests
from IPython.display import Image, display

# Download a sample architecture diagram image
image_url = "https://storage.googleapis.com/cloud-samples-data/ai-platform/flowers/daisy/100080576_f52e8ee070_n.jpg"
image_bytes = requests.get(image_url).content

# Display the image
display(Image(data=image_bytes, width=400))
print(f"Image size: {len(image_bytes)} bytes")

In [ ]:
# Multimodal prompt: analyze the image

image_part = Part.from_data(data=image_bytes, mime_type="image/jpeg")

# Visual QA
response_visual = model.generate_content(
    [
        image_part,
        """Analyze this image and return a JSON object with:
        - "description": 2-3 sentence description
        - "main_subject": what is the primary subject
        - "colors": list of dominant colors
        - "mood": the overall mood of the image
        Return ONLY valid JSON."""
    ],
    generation_config=GenerationConfig(temperature=0.2)
)

print("=== Multimodal Image Analysis ===")
print(response_visual.text)

In [ ]:
# Multimodal: compare two prompting approaches on the same image

# Approach 1: Open-ended
response_open = model.generate_content(
    [image_part, "What do you see?"]
)

# Approach 2: Structured
response_structured = model.generate_content(
    [image_part, """Describe this image using exactly this format:
    Subject: [main subject]
    Setting: [environment/location]
    Details: [2-3 specific observations]
    Suggested use: [how this image could be used commercially]"""]
)

print("=== Open-ended ===")
print(response_open.text)
print("\n=== Structured ===")
print(response_structured.text)

---
## 5. RAG Demo: Embed, Retrieve, Generate

In [ ]:
# RAG Step 1: Create a document collection
# We'll use a simple in-memory approach with chromadb for demonstration

from vertexai.language_models import TextEmbeddingModel

# Initialize the embedding model
embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-004")

# Sample company knowledge base documents
documents = [
    {
        "id": "doc1",
        "text": "Our company vacation policy allows 20 days of PTO per year for full-time employees. "
                "PTO accrues at 1.67 days per month. Unused PTO can be carried over up to 5 days.",
        "source": "hr-policy.pdf"
    },
    {
        "id": "doc2",
        "text": "Remote work is permitted 3 days per week. Employees must be in the office on Tuesdays "
                "and Thursdays. Remote work from outside the country requires VP approval.",
        "source": "hr-policy.pdf"
    },
    {
        "id": "doc3",
        "text": "The company uses Google Cloud Platform for all infrastructure. Production workloads "
                "run on GKE in us-central1. Staging environments use Cloud Run for cost optimization.",
        "source": "infra-guide.pdf"
    },
    {
        "id": "doc4",
        "text": "All ML models must be registered in Vertex AI Model Registry before deployment. "
                "Models require at least 95% accuracy on the test set and bias evaluation approval.",
        "source": "ml-guidelines.pdf"
    },
    {
        "id": "doc5",
        "text": "Health insurance covers employees and dependents. The company pays 80% of premiums. "
                "Dental and vision are included. Open enrollment is in November each year.",
        "source": "benefits-guide.pdf"
    }
]

print(f"Knowledge base: {len(documents)} documents loaded")

In [ ]:
# RAG Step 2: Embed all documents using Vertex AI

texts = [doc["text"] for doc in documents]
embeddings_response = embedding_model.get_embeddings(texts)

doc_embeddings = []
for i, embedding in enumerate(embeddings_response):
    doc_embeddings.append({
        "id": documents[i]["id"],
        "text": documents[i]["text"],
        "source": documents[i]["source"],
        "embedding": embedding.values
    })

print(f"Embedded {len(doc_embeddings)} documents")
print(f"Embedding dimension: {len(doc_embeddings[0]['embedding'])}")

In [ ]:
# RAG Step 3: Retrieve relevant documents using cosine similarity

import numpy as np

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve(query: str, top_k: int = 2):
    """Embed the query and find the most similar documents."""
    query_embedding = embedding_model.get_embeddings([query])[0].values
    
    scores = []
    for doc in doc_embeddings:
        score = cosine_similarity(query_embedding, doc["embedding"])
        scores.append((doc, score))
    
    # Sort by similarity (highest first)
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

# Test retrieval
query = "How many vacation days do I get?"
results = retrieve(query, top_k=2)

print(f"Query: {query}\n")
for doc, score in results:
    print(f"Score: {score:.4f} | Source: {doc['source']}")
    print(f"  {doc['text'][:100]}...\n")

In [ ]:
# RAG Step 4: Generate answer using retrieved context

def rag_generate(query: str, top_k: int = 2):
    """Full RAG pipeline: retrieve context, then generate answer."""
    # Retrieve
    results = retrieve(query, top_k=top_k)
    context_texts = [doc["text"] for doc, _ in results]
    sources = [doc["source"] for doc, _ in results]
    
    # Build the augmented prompt
    context = "\n\n".join(context_texts)
    prompt = f"""Answer the question based ONLY on the provided context.
If the context doesn't contain enough information, say "I don't have enough information."
Always cite the source document.

Context:
{context}

Question: {query}

Answer:"""
    
    # Generate
    response = model.generate_content(
        prompt,
        generation_config=GenerationConfig(temperature=0.1)
    )
    
    return response.text, sources

# Test the full RAG pipeline
questions = [
    "How many vacation days do I get per year?",
    "Can I work remotely from another country?",
    "Where do we deploy production ML models?",
    "What is the stock option vesting schedule?"  # Not in our docs
]

for q in questions:
    answer, sources = rag_generate(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print(f"Sources: {sources}\n")

### RAG Pipeline Summary

| Step | What Happens | GCP Service |
|------|-------------|-------------|
| **Embed** | Convert docs to vectors | Vertex AI Embeddings API |
| **Store** | Persist vectors | Vertex AI Vector Search, AlloyDB, Spanner |
| **Retrieve** | Similarity search | ANN index query |
| **Generate** | LLM + context | Gemini via Vertex AI |

**Production note**: For real applications, use Vertex AI Vector Search instead of in-memory similarity for scalable ANN search.

---
## 6. Grounding with Google Search

In [ ]:
# Grounding attaches real-time search results to model responses
# This is a platform-level feature, not a custom RAG pipeline

from vertexai.preview.generative_models import grounding

# Create a Google Search grounding tool
search_tool = Tool.from_google_search_retrieval(
    grounding.GoogleSearchRetrieval()
)

grounded_model = GenerativeModel("gemini-1.5-pro")

# Query that benefits from real-time information
response_grounded = grounded_model.generate_content(
    "What are the latest Google Cloud AI announcements from Cloud Next 2024?",
    tools=[search_tool]
)

print("=== Grounded Response (Google Search) ===")
print(response_grounded.text)

# Check for grounding metadata
if hasattr(response_grounded.candidates[0], 'grounding_metadata'):
    gm = response_grounded.candidates[0].grounding_metadata
    print("\n=== Grounding Sources ===")
    if hasattr(gm, 'search_entry_point'):
        print(f"Search query used: {gm.search_entry_point.rendered_content[:200]}")

In [ ]:
# Compare: same question WITHOUT grounding

response_ungrounded = model.generate_content(
    "What are the latest Google Cloud AI announcements from Cloud Next 2024?"
)

print("=== Ungrounded Response ===")
print(response_ungrounded.text)
print("\n--- Notice: The ungrounded response may contain outdated or hallucinated information ---")

### Grounding vs RAG

| Feature | Google Search Grounding | Custom RAG |
|---------|----------------------|------------|
| **Data source** | Public web | Your private documents |
| **Setup** | One API parameter | Build full pipeline |
| **Customization** | Limited | Full control |
| **Citations** | Automatic | You implement |
| **Best for** | Current events, public facts | Internal knowledge bases |

---
## 7. Vertex AI Agent Builder Concepts

In [ ]:
# Agent Builder is primarily configured via the Console or API
# Here we demonstrate the SDK concepts for programmatic access

# Note: This requires an existing Agent Builder data store
# The code below shows the pattern; replace IDs with your actual resources

from google.cloud import discoveryengine_v1 as discoveryengine

# --- Data Store Creation (conceptual) ---
# In practice, create via Console or Terraform
print("=== Agent Builder Architecture ===")
print()
print("1. DATA STORES")
print("   - Ingest: Cloud Storage, BigQuery, Websites, API")
print("   - Auto: Chunking, Embedding, Indexing")
print("   - Types: Unstructured (PDFs), Structured (tables), Website")
print()
print("2. SEARCH APPS")
print("   - Natural language search over data stores")
print("   - Auto-generated summaries with citations")
print("   - Faceted filtering and relevance tuning")
print()
print("3. CONVERSATION AGENTS")
print("   - Multi-turn dialogue grounded in data stores")
print("   - Intent detection and flow control")
print("   - Tool use via webhooks")
print("   - Human agent handoff")
print()
print("4. GENERATIVE APPS")
print("   - RAG-powered Q&A")
print("   - Content generation from data stores")
print("   - Recommendations")

In [ ]:
# --- Programmatic search via Agent Builder API ---
# Uncomment and configure with your actual data store

# from google.cloud import discoveryengine_v1 as discoveryengine
#
# client = discoveryengine.SearchServiceClient()
#
# # Build the search request
# request = discoveryengine.SearchRequest(
#     serving_config=f"projects/{PROJECT_ID}/locations/global/collections/default_collection/"
#                    f"dataStores/my-data-store/servingConfigs/default_serving_config",
#     query="What is our deployment process?",
#     page_size=5,
#     content_search_spec=discoveryengine.SearchRequest.ContentSearchSpec(
#         snippet_spec=discoveryengine.SearchRequest.ContentSearchSpec.SnippetSpec(
#             return_snippet=True
#         ),
#         summary_spec=discoveryengine.SearchRequest.ContentSearchSpec.SummarySpec(
#             summary_result_count=3,
#             include_citations=True
#         )
#     )
# )
#
# response = client.search(request)
# print(f"Summary: {response.summary.summary_text}")
# for result in response.results:
#     print(f"  - {result.document.derived_struct_data.get('title', 'N/A')}")

print("Agent Builder search API pattern demonstrated (uncomment with real data store ID to run).")

---
## 8. Deploy a GenAI App on Cloud Run

In [ ]:
# This cell generates the deployment files for a simple GenAI Flask app on Cloud Run
# You would run these locally or in Cloud Shell, not in this notebook

# --- app.py ---
app_py = '''import os
from flask import Flask, request, jsonify
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig

app = Flask(__name__)

# Initialize Vertex AI
PROJECT_ID = os.environ.get("PROJECT_ID", "my-project")
LOCATION = os.environ.get("LOCATION", "us-central1")
vertexai.init(project=PROJECT_ID, location=LOCATION)

model = GenerativeModel(
    "gemini-1.5-pro",
    system_instruction="You are a helpful assistant. Be concise and accurate."
)

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "healthy"})

@app.route("/generate", methods=["POST"])
def generate():
    data = request.get_json()
    prompt = data.get("prompt", "")
    temperature = data.get("temperature", 0.3)
    
    if not prompt:
        return jsonify({"error": "prompt is required"}), 400
    
    config = GenerationConfig(
        temperature=temperature,
        max_output_tokens=1024
    )
    response = model.generate_content(prompt, generation_config=config)
    
    return jsonify({
        "response": response.text,
        "token_count": response.usage_metadata.total_token_count
    })

if __name__ == "__main__":
    port = int(os.environ.get("PORT", 8080))
    app.run(host="0.0.0.0", port=port)
'''

# --- Dockerfile ---
dockerfile = '''FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
ENV PORT=8080
CMD ["gunicorn", "--bind", "0.0.0.0:8080", "--workers", "2", "--timeout", "120", "app:app"]
'''

# --- requirements.txt ---
requirements = '''flask>=3.0.0
gunicorn>=21.2.0
google-cloud-aiplatform>=1.60.0
'''

# --- Deploy command ---
deploy_cmd = f'''# Build and deploy to Cloud Run
gcloud run deploy genai-app \\
  --source . \\
  --region {LOCATION} \\
  --service-account genai-sa@{PROJECT_ID}.iam.gserviceaccount.com \\
  --set-env-vars PROJECT_ID={PROJECT_ID},LOCATION={LOCATION} \\
  --memory 1Gi \\
  --timeout 120 \\
  --allow-unauthenticated
'''

print("=== app.py ===")
print(app_py)
print("=== Dockerfile ===")
print(dockerfile)
print("=== requirements.txt ===")
print(requirements)
print("=== Deploy Command ===")
print(deploy_cmd)

In [ ]:
# Test the deployed endpoint (after deployment)

# import requests
#
# SERVICE_URL = "https://genai-app-xxxxx-uc.a.run.app"  # Replace with your Cloud Run URL
#
# # Health check
# health = requests.get(f"{SERVICE_URL}/health")
# print(f"Health: {health.json()}")
#
# # Generate
# response = requests.post(
#     f"{SERVICE_URL}/generate",
#     json={"prompt": "Explain Vertex AI in 2 sentences.", "temperature": 0.2}
# )
# print(f"Response: {response.json()}")

print("Cloud Run deployment pattern demonstrated.")
print("Uncomment and replace SERVICE_URL after deploying to test.")

---
## Summary

In this notebook, you learned:

1. **Prompt Design**: System instructions, zero-shot, few-shot, chain-of-thought, and structured output
2. **Function Calling**: Define tools, handle model function calls, complete the round-trip
3. **Multimodal**: Analyze images with Gemini using text + image prompts
4. **RAG Pipeline**: Embed documents with Vertex AI, retrieve by similarity, generate grounded answers
5. **Grounding**: Use Google Search grounding for real-time factual responses
6. **Agent Builder**: Understand data stores, search apps, and conversation agents
7. **Deployment**: Package and deploy a GenAI app on Cloud Run with Dockerfile and gcloud commands

**Next**: [Course 18 - Responsible AI: Fairness and Bias](18-responsible-ai-fairness.ipynb)